# 02 — Entrenamiento en Google Colab GPU (Fase 3: Jueces + Pretrain Pix2Pix)

Este notebook ejecuta en GPU (**Entorno de ejecución → Cambiar tipo de entorno → T4 GPU**):
1. Montaje de **Google Drive** para persistir `checkpoints/`, `results/` y `figuras/`.
2. Clonación / actualización del repositorio `handwritten-font-cgan`.
3. Generación reproducible del dataset sintético (`SEED = 42`).
4. Entrenamiento de los clasificadores jueces congelables (**ResNet-18** y **EfficientNet-B0**).
5. Preentrenamiento de **Pix2Pix (U-Net + PatchGAN)** sobre pares sintéticos.
6. Verificación visual y cuantitativa del **Checkpoint 3**.

In [ ]:
# 1. Verificar GPU y montar Google Drive
from pathlib import Path
import torch
from google.colab import drive

print("PyTorch:", torch.__version__, "| CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("ADVERTENCIA: No hay GPU activa. Activa GPU en Entorno de ejecución -> Cambiar tipo de entorno.")

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/handwritten-font-cgan")
CKPT_DIR = DRIVE_ROOT / "checkpoints"
FIG_DIR = DRIVE_ROOT / "figuras"
RES_DIR = DRIVE_ROOT / "results"
for d in (CKPT_DIR, FIG_DIR, RES_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Artefactos persistentes en:", DRIVE_ROOT)

In [ ]:
# 2. Clonar o actualizar el repositorio desde GitHub
import os
import subprocess

REPO_URL = "https://github.com/carloamezcua/handwritten-font-cgan.git"
REPO_DIR = Path("/content/handwritten-font-cgan")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("Directorio actual:", os.getcwd())

# Instalar solo librerías no incluidas por defecto en Colab (sin reinstalar PyTorch CUDA)
!pip install -q fonttools

In [ ]:
# 3. Generar dataset sintético en Colab (idéntico al local gracias a SEED=42)
if not Path("data/synthetic_pairs/metadata.csv").exists():
    !python -m src.data.03_generar_sinteticos --download-fonts
else:
    print("Dataset sintético ya existe en data/synthetic_pairs/")

In [ ]:
# 4. Entrenar Jueces (ResNet-18 y EfficientNet-B0) con Early Stopping
!python -m src.train.train_judges \
    --model all \
    --data-dir data/synthetic_pairs \
    --checkpoints-dir "{CKPT_DIR}" \
    --figures-dir "{FIG_DIR}" \
    --results-dir "{RES_DIR / 'judges'}" \
    --epochs 30 \
    --batch-size 64 \
    --lr 1e-3 \
    --patience 7 \
    --num-workers 2 \
    --pretrained

In [ ]:
# 5. Checkpoint 3 (Parte A): Verificar que ambos jueces superan 90% Macro-F1
import json
from IPython.display import Image as IPyImage, display

for name in ("resnet18", "efficientnet_b0"):
    summary_path = RES_DIR / "judges" / f"judge_{name}_history.json"
    data = json.loads(summary_path.read_text(encoding="utf-8"))
    f1 = data["best_val_macro_f1"]
    acc = data["best_val_acc"]
    status = "OK (>= 0.90)" if f1 >= 0.90 else "REVISAR (< 0.90)"
    print(f"{name:16s} | época={data['best_epoch']:02d} | val_acc={acc:.4f} | val_macro_f1={f1:.4f} -> {status}")
    display(IPyImage(filename=str(FIG_DIR / f"train_judge_{name}.png")))

In [ ]:
# 6. Preentrenar Pix2Pix sobre el dataset sintético (64 símbolos en train, 11 reservados en test_unseen)
!python -m src.train.train_pix2pix \
    --mode pretrain \
    --data-dir data/synthetic_pairs \
    --checkpoints-dir "{CKPT_DIR}" \
    --figures-dir "{FIG_DIR}" \
    --results-dir "{RES_DIR / 'pix2pix'}" \
    --epochs 60 \
    --batch-size 32 \
    --lr 2e-4 \
    --sample-every 10 \
    --num-workers 2

In [ ]:
# 7. Checkpoint 3 (Parte B): Inspección visual de curvas y cuadrículas x | ŷ | y
for fname in (
    "train_pix2pix_pretrain.png",
    "pix2pix_pretrain_val_grid.png",
    "pix2pix_pretrain_unseen_grid.png",
):
    print(f"\n--- {fname} ---")
    display(IPyImage(filename=str(FIG_DIR / fname)))

## Siguiente paso al terminar en Colab
1. Desde tu Google Drive (`MyDrive/handwritten-font-cgan/checkpoints/`), descarga a tu carpeta local `checkpoints/`:
   - `judge_resnet18.pth`
   - `judge_efficientnet_b0.pth`
   - `pix2pix_pretrained.pth`
2. Desde `MyDrive/handwritten-font-cgan/figuras/`, copia a tu carpeta local `docs/figuras/`:
   - `train_judge_resnet18.png`, `train_judge_efficientnet_b0.png`
   - `train_pix2pix_pretrain.png`, `pix2pix_pretrain_val_grid.png`, `pix2pix_pretrain_unseen_grid.png`
3. Haz `git add docs/figuras/` y commit de las gráficas (los `.pth` en `checkpoints/` quedan ignorados por Git).